# Deepfake Detection & Attribution con Explainability

**Multimedia Forensics — Laurea Magistrale**  ·  pipeline multi-stream (RGB + Fourier) su Colab T4.

Pipeline:
1. **Feature**: immagine → ResNet18(ImageNet) → embedding ; spettro di Fourier → ResNet18 → embedding
2. **Classificatore a cascata** sugli embedding concatenati → *detection* (real/fake); se *fake* → *attribution* tra i generatori
3. **Explainability**: Grad-CAM sui due stream + **agent VLM open** (Qwen2.5-VL) che spiega il *perché*

> `Runtime → Change runtime type → T4 GPU` prima di eseguire.

## 1. Setup

In [ ]:
# Installazione dipendenze (su Colab torch/torchvision ci sono già).
# Decommenta alla prima esecuzione su Colab:
# !pip -q install transformers>=4.49 accelerate bitsandbytes qwen-vl-utils sentencepiece pyyaml

import torch
print('torch', torch.__version__, '| CUDA:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))

In [ ]:
# Rendi importabile il package `dffa`.
# Su Colab: carica/clona la cartella Progetto e imposta il percorso, es.:
#   from google.colab import drive; drive.mount('/content/drive')
#   %cd '/content/drive/MyDrive/Progetto'
import sys, os
# Se esegui il notebook dentro Progetto/notebooks, aggiungi la root del progetto:
ROOT = os.path.abspath('..') if os.path.basename(os.getcwd()) == 'notebooks' else os.getcwd()
if ROOT not in sys.path:
    sys.path.insert(0, ROOT)
print('project root:', ROOT)

import dffa
print('dffa', dffa.__version__)

## 2. Configurazione

In [ ]:
from dffa.config import Config
from dffa.utils import set_seed, get_device, ensure_dir, save_json

cfg = Config(
    data_root=os.path.join(ROOT, 'data'),
    classes=['real', 'stylegan', 'stylegan3', 'sdxl'],
    max_per_class=250,      # ~1000 immagini totali; metti None per usarle tutte
    epochs=30,
    batch_size=32,
)
set_seed(cfg.seed)
device = get_device(cfg.device)
results_dir = ensure_dir(os.path.join(ROOT, cfg.results_dir))
print('device:', device, '| classi:', cfg.classes)

## 3. Dati

Disponi le immagini in `data/<classe>/` (es. `data/real`, `data/stylegan`, …).
Le classi assenti vengono ignorate: puoi partire anche con sole 2 classi.

In [ ]:
from dffa.data import build_splits, list_samples
from collections import Counter

all_samples = list_samples(cfg)
if not all_samples:
    raise SystemExit('Nessuna immagine trovata: popola data/<classe>/ prima di proseguire.')
counts = Counter(cfg.classes[i] for _, i in all_samples)
print('immagini per classe:', dict(counts))

splits = build_splits(cfg)
print({k: len(v) for k, v in splits.items()})

## 4. Feature extraction (ResNet18 dual-stream, con cache)

In [ ]:
from dffa.features import DualStreamExtractor
from dffa.engine import extract_embeddings

extractor = DualStreamExtractor(pretrained=True, freeze=True).to(device)

blobs = {}
for name in ['train', 'val', 'test']:
    cache = os.path.join(results_dir, f'emb_{name}.pt')
    blobs[name] = extract_embeddings(splits[name], cfg, extractor, device, cache_path=cache)
    print(name, blobs[name]['embeddings'].shape)

## 5. Sanity check — immagine vs spettro di Fourier

In [ ]:
import matplotlib.pyplot as plt
from PIL import Image
from dffa.features.fourier import fourier_spectrum

def show_pair(path, title):
    img = Image.open(path).convert('RGB')
    spec = fourier_spectrum(img, log=True)
    fig, ax = plt.subplots(1, 2, figsize=(7, 3.4))
    ax[0].imshow(img); ax[0].set_title(f'{title} — RGB'); ax[0].axis('off')
    ax[1].imshow(spec, cmap='inferno'); ax[1].set_title('Spettro di Fourier (log)'); ax[1].axis('off')
    plt.tight_layout(); plt.show()

# un esempio reale e uno fake (se disponibili)
by_cls = {}
for p, i in all_samples:
    by_cls.setdefault(cfg.classes[i], p)
for cls in ['real', 'stylegan']:
    if cls in by_cls:
        show_pair(by_cls[cls], cls)

## 6. Training del classificatore multi-task

In [ ]:
from dffa.engine import train_classifier

model, history = train_classifier(blobs['train'], blobs['val'], cfg, device)

fig, ax = plt.subplots(1, 2, figsize=(10, 3.6))
ax[0].plot(history['train_loss'], label='train'); ax[0].plot(history['val_loss'], label='val')
ax[0].set_title('Loss'); ax[0].set_xlabel('epoch'); ax[0].legend()
ax[1].plot(history['val_det_acc'], label='detection'); ax[1].plot(history['val_attr_acc'], label='attribution (fake)')
ax[1].plot(history['val_cascade_acc'], label='cascade')
ax[1].set_title('Val accuracy'); ax[1].set_xlabel('epoch'); ax[1].legend()
plt.tight_layout(); plt.show()

## 7. Valutazione sul test set

In [ ]:
from dffa.engine import evaluate, _loader_from_blob
from sklearn.metrics import classification_report, confusion_matrix
import numpy as np

test_loader = _loader_from_blob(blobs['test'], cfg, shuffle=False)
res = evaluate(model, test_loader, cfg, device)
print(f"Detection acc          : {res['detection_acc']:.4f}")
print(f"Attribution acc (fake) : {res['attribution_acc']:.4f}")
print(f"Cascade acc (end-to-end): {res['cascade_acc']:.4f}")

print('\n== Detection (real vs fake, tutti i campioni) ==')
print(classification_report(res['detection_true'], res['detection_pred'],
                            target_names=['real', 'fake'], zero_division=0))
print('== Attribution (SOLO fake, tra generatori) ==')
gens = cfg.generator_classes
present = sorted(set(res['attribution_true']) | set(res['attribution_pred']))
print(classification_report(res['attribution_true'], res['attribution_pred'],
                            labels=present, target_names=[gens[i] for i in present],
                            zero_division=0))

cm = confusion_matrix(res['attribution_true'], res['attribution_pred'], labels=present)
fig, ax = plt.subplots(figsize=(4.6, 4))
im = ax.imshow(cm, cmap='Blues')
ax.set_xticks(range(len(present))); ax.set_xticklabels([gens[i] for i in present], rotation=45, ha='right')
ax.set_yticks(range(len(present))); ax.set_yticklabels([gens[i] for i in present])
ax.set_xlabel('pred'); ax.set_ylabel('true'); ax.set_title('Confusion matrix (attribution, solo fake)')
for r in range(cm.shape[0]):
    for c in range(cm.shape[1]):
        ax.text(c, r, cm[r, c], ha='center', va='center')
plt.colorbar(im); plt.tight_layout(); plt.show()

## 8. Grad-CAM — dove guarda la rete (RGB e Fourier)

In [ ]:
from dffa.data import ForensicsDataset
from dffa.explain import GradCAM, overlay_cam
from dffa.models import cascade_predict

# scelgo un campione FAKE dal test, così l'attribution è applicabile
sample_path, sample_attr = next((s for s in splits['test'] if cfg.classes[s[1]] != 'real'), splits['test'][0])
ds_one = ForensicsDataset([(sample_path, sample_attr)], cfg)
item = ds_one[0]
rgb = item['rgb'].unsqueeze(0).to(device)
fft = item['fourier'].unsqueeze(0).to(device)

# embedding + decisione a cascata (detection -> se fake -> attribution)
with torch.no_grad():
    emb = extractor(rgb, fft)
    pred, decisions = cascade_predict(model, emb, cfg.generator_classes)
dec = decisions[0]
det_cls = int(pred['detection_pred'][0]); gen_cls = int(pred['attribution_pred'][0])
print('detection:', dec['detection'], '| attribution:', dec['attribution'])

# Grad-CAM sullo stream RGB rispetto al logit di detection predetto
cam_rgb = GradCAM(extractor.rgb, extractor.rgb.target_layer)
heat_rgb = cam_rgb(rgb, score_fn=lambda f: model(torch.cat([f, extractor.fourier(fft)],1)).detection_logits[:, det_cls])
cam_rgb.remove()

# Grad-CAM sullo stream Fourier rispetto al generatore predetto (attribution)
cam_fft = GradCAM(extractor.fourier, extractor.fourier.target_layer)
heat_fft = cam_fft(fft, score_fn=lambda f: model(torch.cat([extractor.rgb(rgb), f],1)).attribution_logits[:, gen_cls])
cam_fft.remove()

ov_rgb = overlay_cam(item['rgb'], heat_rgb)
ov_fft = overlay_cam(item['fourier'], heat_fft)
fig, ax = plt.subplots(1, 2, figsize=(7.5, 3.8))
ax[0].imshow(ov_rgb); ax[0].set_title('Grad-CAM RGB (detection)'); ax[0].axis('off')
ax[1].imshow(ov_fft); ax[1].set_title('Grad-CAM Fourier (attribution)'); ax[1].axis('off')
plt.tight_layout(); plt.show()

## 9. Agent VLM — spiegazione in linguaggio naturale

L'agent open (Qwen2.5-VL, 4-bit su T4) osserva RGB + Fourier + Grad-CAM e le
probabilità del classificatore, e spiega *perché* real/fake e *perché* quel
generatore. Senza GPU/modello ricade su una spiegazione template-based.

In [ ]:
from dffa.explain import VLMExplainer, build_evidence
from dffa.explain.gradcam import _jet
from PIL import Image as PILImage

explainer = VLMExplainer(cfg.vlm_model_id, load_in_4bit=cfg.vlm_load_in_4bit,
                         max_new_tokens=cfg.vlm_max_new_tokens, device=str(device))
loaded = explainer.load() if cfg.vlm_enabled else False
print('VLM caricato:', loaded)

# evidenze numeriche per il campione di Grad-CAM
evidence = build_evidence(
    detection_prob=pred['detection_prob'][0].cpu().numpy(),
    attribution_prob=pred['attribution_prob'][0].cpu().numpy(),
    generator_classes=cfg.generator_classes,
    cam_stats={'mean': float(heat_rgb.mean()), 'max': float(heat_rgb.max())},
)

images = {
    'rgb': PILImage.open(sample_path).convert('RGB'),
    'fourier': PILImage.fromarray((_jet(fourier_spectrum(PILImage.open(sample_path).convert('RGB')))*255).astype('uint8')),
    'gradcam': PILImage.fromarray(ov_rgb),
}
exp = explainer.explain(images, evidence)
print(f'[fonte: {exp.source}]\n')
print(exp.text)

## 10. Salvataggio dei risultati

In [ ]:
summary = {
    'config': cfg.__dict__,
    'counts': dict(counts),
    'splits': {k: len(v) for k, v in splits.items()},
    'detection_acc': res['detection_acc'],
    'attribution_acc': res['attribution_acc'],
    'cascade_acc': res['cascade_acc'],
    'example_explanation': {'path': sample_path, 'detection': dec['detection'],
                            'attribution': dec['attribution'], 'source': exp.source, 'text': exp.text},
}
save_json(summary, os.path.join(results_dir, 'summary.json'))
torch.save(model.state_dict(), os.path.join(results_dir, 'classifier.pt'))
print('Salvato in', results_dir)